# Análise por gênero

Já sei o que separa hits de não-hits no geral. Agora quero entender se isso muda entre gêneros. Pode ser que danceability alta funcione pra pop mas não pra metal, ou que acousticness importa pra folk mas não pra eletrônica. Se a gravadora quer apostar num artista de um gênero específico, precisa saber o perfil de hit *daquele* gênero.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv("../data/processed/tracks_clean.csv")

FEATURES = [
    "danceability", "energy", "valence", "tempo",
    "loudness", "speechiness", "acousticness",
    "instrumentalness", "duration_ms",
]

print(f"Total de gêneros: {df['track_genre'].nunique()}")
print(f"Total de hits: {df['is_hit'].sum()}")

Total de gêneros: 113
Total de hits: 8834


## Quais gêneros produzem mais hits?

Se certos gêneros concentram a maioria dos hits, isso já é uma informação útil. A gravadora pode estar apostando num gênero que raramente estoura.

In [2]:
hits_by_genre = (
    df.groupby("track_genre")["is_hit"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "hits", "count": "total"})
)
hits_by_genre["hit_rate"] = (hits_by_genre["hits"] / hits_by_genre["total"] * 100).round(1)
hits_by_genre = hits_by_genre.sort_values("hit_rate", ascending=False).reset_index()

top15 = hits_by_genre.head(15)

fig = px.bar(
    top15, x="hit_rate", y="track_genre",
    orientation="h", text="hit_rate",
    title="Top 15 gêneros com maior taxa de hit (%)",
    labels={"hit_rate": "Taxa de hit (%)", "track_genre": ""},
)
fig.update_layout(height=500, template="plotly_white", yaxis=dict(autorange="reversed"))
fig.show()

In [3]:
bottom15 = hits_by_genre.tail(15)

fig = px.bar(
    bottom15, x="hit_rate", y="track_genre",
    orientation="h", text="hit_rate",
    title="15 gêneros com menor taxa de hit (%)",
    labels={"hit_rate": "Taxa de hit (%)", "track_genre": ""},
    color_discrete_sequence=["#EF553B"],
)
fig.update_layout(height=500, template="plotly_white")
fig.show()

## Perfil sonoro dos gêneros que mais produzem hits

Pego os 5 gêneros com mais hits e comparo o perfil médio de cada um. Isso responde: "o que faz um hit em pop é diferente do que faz um hit em hip-hop?"

In [4]:
top5_genres = hits_by_genre.head(5)["track_genre"].tolist()
hits_top5 = df[(df["is_hit"] == 1) & (df["track_genre"].isin(top5_genres))]

radar_feats = [f for f in FEATURES if f != "duration_ms"]

fig = go.Figure()
for genre in top5_genres:
    genre_data = hits_top5[hits_top5["track_genre"] == genre][radar_feats].mean()
    fig.add_trace(go.Scatterpolar(
        r=genre_data.values, theta=radar_feats,
        fill="toself", name=genre, opacity=0.5,
    ))

fig.update_layout(
    title="Perfil sonoro dos hits — top 5 gêneros",
    polar=dict(radialaxis=dict(visible=True)),
    height=550, template="plotly_white",
)
fig.show()

## Como cada feature se comporta entre gêneros

Comparo as médias de danceability, energy, acousticness e speechiness nos 10 gêneros com mais hits. Se as barras forem parecidas, a feature não distingue gêneros. Se forem bem diferentes, o gênero importa na hora de avaliar aquela feature.

In [5]:
top10_genres = hits_by_genre.head(10)["track_genre"].tolist()
hits_top10 = df[(df["is_hit"] == 1) & (df["track_genre"].isin(top10_genres))]

key_features = ["danceability", "energy", "acousticness", "speechiness"]

fig = make_subplots(rows=2, cols=2, subplot_titles=key_features)

for i, feat in enumerate(key_features):
    row, col = i // 2 + 1, i % 2 + 1
    genre_means = hits_top10.groupby("track_genre")[feat].mean().sort_values(ascending=False)
    fig.add_trace(
        go.Bar(x=genre_means.index, y=genre_means.values, showlegend=False),
        row=row, col=col,
    )

fig.update_layout(height=700, template="plotly_white", title_text="Features dos hits por gênero")
fig.show()

## Hits vs não-hits dentro de um mesmo gênero

Pego o gênero com mais tracks e comparo hits vs não-hits ali dentro. Isso mostra se as mesmas features que separam classes no geral continuam separando dentro de um gênero específico.

In [6]:
biggest_genre = df["track_genre"].value_counts().index[0]
genre_df = df[df["track_genre"] == biggest_genre].copy()
genre_df["is_hit_label"] = genre_df["is_hit"].map({0: "Não-hit", 1: "Hit"})

print(f"Gênero: {biggest_genre} ({len(genre_df)} tracks, {genre_df['is_hit'].sum()} hits)\n")

fig = make_subplots(rows=3, cols=3, subplot_titles=FEATURES)

for i, feat in enumerate(FEATURES):
    row, col = i // 3 + 1, i % 3 + 1
    for label, color in [("Não-hit", "#636EFA"), ("Hit", "#EF553B")]:
        subset = genre_df[genre_df["is_hit_label"] == label]
        fig.add_trace(
            go.Box(
                y=subset[feat], name=label,
                marker_color=color, showlegend=(i == 0),
                legendgroup=label,
            ),
            row=row, col=col,
        )

fig.update_layout(
    height=900, template="plotly_white",
    title_text=f"Hits vs Não-hits dentro do gênero: {biggest_genre}",
)
fig.show()

Gênero: study (996 tracks, 0 hits)



## Conclusões

**Quais gêneros produzem mais hits?**

Pop, hip-hop e dance lideram em taxa de hit. Gêneros de nicho como grindcore, black metal e study music quase não aparecem nos hits. Se a gravadora quer apostar no topo dos charts, o gênero de entrada já filtra boa parte das chances.

**O perfil de um hit muda entre gêneros?**

Bastante. Um hit de hip-hop tem speechiness alta e danceability forte. Um hit de rock tem energy e loudness no topo mas speechiness baixa. Usar o mesmo critério pra todos os gêneros é um erro — o modelo vai precisar aprender essas diferenças.

**O que isso significa na prática?**

Se um artista indie mandar uma demo acústica com 3 minutos, comparar com a média geral de hits não faz sentido. O certo é comparar com o perfil de hit *daquele gênero*. O app deveria levar isso em conta — ou pelo menos a gravadora deveria, na hora de interpretar o resultado.